In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ishaans23/segmentation-training-artifact/training_history.csv
/kaggle/input/datasets/ishaans23/segmentation-training-artifact/training_summary.json
/kaggle/input/datasets/ishaans23/segmentation-training-artifact/unet_best.pth
/kaggle/input/datasets/ishaans23/final-classifier-artifact/head_training_history.csv
/kaggle/input/datasets/ishaans23/final-classifier-artifact/layer4_training_history.csv
/kaggle/input/datasets/ishaans23/final-classifier-artifact/model_metadata.json
/kaggle/input/datasets/ishaans23/final-classifier-artifact/resnet50_layer4_best.pth
/kaggle/input/datasets/ishaans23/final-classifier-artifact/training_summary.json
/kaggle/input/datasets/ishaans23/cbis-ddsm-training/cache_info.json
/kaggle/input/datasets/ishaans23/cbis-ddsm-training/excluded_mass_cases.csv
/kaggle/input/datasets/ishaans23/cbis-ddsm-training/image_metadata.csv
/kaggle/input/datasets/ishaans23/cbis-ddsm-training/shared_mass_calc_audit.csv
/kaggle/input/datasets/ishaans23/cbis-dds

In [2]:
from pathlib import Path
import json
import random

import cv2
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50
from tqdm.auto import tqdm

In [3]:
SEED = 42

TARGET_HEIGHT = 1024
TARGET_WIDTH = 640

BATCH_SIZE = 1
NUM_WORKERS = 0

MAX_EPOCHS = 8
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_MIN_DELTA = 1e-4

DECODER_LR = 3e-4
WEIGHT_DECAY = 1e-5

BCE_WEIGHT = 0.30
DICE_WEIGHT = 0.70

USE_AMP = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUTPUT_DIR = Path(
    "/kaggle/working/resnet50_unet_training"
)

CHECKPOINT_DIR = (
    OUTPUT_DIR / "checkpoints"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

BEST_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "resnet50_unet_decoder_best.pth"
)

LATEST_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "resnet50_unet_decoder_latest.pth"
)

HISTORY_PATH = (
    OUTPUT_DIR
    / "decoder_training_history.csv"
)

print("Output:", OUTPUT_DIR)

Output: /kaggle/working/resnet50_unet_training


In [4]:
KAGGLE_INPUT_ROOT = Path(
    "/kaggle/input"
)

dataset_candidates = []

for cache_info_path in KAGGLE_INPUT_ROOT.rglob(
    "cache_info.json"
):

    candidate_root = (
        cache_info_path.parent
    )

    required = [
        candidate_root / "npy",
        candidate_root / "masks",
        candidate_root / "splits",
    ]

    if all(
        path.exists()
        for path in required
    ):
        dataset_candidates.append(
            candidate_root
        )

assert len(dataset_candidates) > 0

dataset_candidates = sorted(
    dataset_candidates,
    key=lambda path: len(str(path)),
)

PROJECT_ROOT = dataset_candidates[0]

print("Dataset:")
print(PROJECT_ROOT)

Dataset:
/kaggle/input/datasets/ishaans23/cbis-ddsm-training


In [5]:
checkpoint_candidates = list(
    KAGGLE_INPUT_ROOT.rglob(
        "resnet50_layer4_best.pth"
    )
)

metadata_candidates = list(
    KAGGLE_INPUT_ROOT.rglob(
        "model_metadata.json"
    )
)

assert len(checkpoint_candidates) > 0, (
    "Classifier checkpoint not found."
)

assert len(metadata_candidates) > 0, (
    "Classifier metadata not found."
)

CLASSIFIER_CHECKPOINT_PATH = (
    checkpoint_candidates[0]
)

CLASSIFIER_METADATA_PATH = (
    metadata_candidates[0]
)

with open(
    CLASSIFIER_METADATA_PATH,
    "r",
) as f:

    classifier_metadata = json.load(f)

print("Classifier checkpoint:")
print(CLASSIFIER_CHECKPOINT_PATH)

print("\nMetadata:")
print(CLASSIFIER_METADATA_PATH)

Classifier checkpoint:
/kaggle/input/datasets/ishaans23/final-classifier-artifact/resnet50_layer4_best.pth

Metadata:
/kaggle/input/datasets/ishaans23/final-classifier-artifact/model_metadata.json


In [6]:
SPLITS_DIR = (
    PROJECT_ROOT / "splits"
)

train_df = pd.read_csv(
    SPLITS_DIR / "train.csv"
)

val_df = pd.read_csv(
    SPLITS_DIR / "val.csv"
)

print("Train:", len(train_df))
print("Validation:", len(val_df))

Train: 1885
Validation: 408


In [7]:
class RAMSegmentationDataset(Dataset):

    def __init__(
        self,
        images,
        masks,
        image_keys,
    ):

        self.images = images
        self.masks = masks
        self.image_keys = image_keys

    def __len__(self):
        return len(self.image_keys)

    def __getitem__(self, idx):

        return (
            self.images[idx],
            self.masks[idx],
            self.image_keys[idx],
        )

In [8]:
def load_split_into_ram(
    dataframe,
    root_dir,
    split_name,
):

    n = len(dataframe)

    images = np.empty(
        (
            n,
            TARGET_HEIGHT,
            TARGET_WIDTH,
        ),
        dtype=np.uint16,
    )

    masks = np.empty(
        (
            n,
            TARGET_HEIGHT,
            TARGET_WIDTH,
        ),
        dtype=np.uint8,
    )

    image_keys = (
        dataframe["image_key"]
        .tolist()
    )


    for i, row in enumerate(
        tqdm(
            dataframe.itertuples(
                index=False
            ),
            total=n,
            desc=f"Loading {split_name}",
        )
    ):

        image = np.load(
            root_dir / row.npy_path
        )

        mask = cv2.imread(
            str(
                root_dir
                / row.mask_path
            ),
            cv2.IMREAD_GRAYSCALE,
        )

        assert image.shape == (
            TARGET_HEIGHT,
            TARGET_WIDTH,
        )

        assert mask is not None

        assert mask.shape == (
            TARGET_HEIGHT,
            TARGET_WIDTH,
        )

        images[i] = image

        masks[i] = (
            mask > 0
        ).astype(
            np.uint8
        )


    return RAMSegmentationDataset(
        images=torch.from_numpy(
            images
        ),
        masks=torch.from_numpy(
            masks
        ),
        image_keys=image_keys,
    )

In [9]:
train_dataset = load_split_into_ram(
    train_df,
    PROJECT_ROOT,
    "Training",
)

val_dataset = load_split_into_ram(
    val_df,
    PROJECT_ROOT,
    "Validation",
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Loading Training:   0%|          | 0/1885 [00:00<?, ?it/s]

Loading Validation:   0%|          | 0/408 [00:00<?, ?it/s]

Train batches: 1885
Validation batches: 408


In [10]:
foreground_pixels = int(
    train_dataset.masks.sum().item()
)

total_pixels = (
    train_dataset.masks.numel()
)

background_pixels = (
    total_pixels
    - foreground_pixels
)

raw_pos_weight = (
    background_pixels
    / foreground_pixels
)

POS_WEIGHT = float(
    np.sqrt(
        raw_pos_weight
    )
)

print(
    "Foreground fraction:",
    foreground_pixels / total_pixels
)

print(
    "Raw positive weight:",
    raw_pos_weight
)

print(
    "Used positive weight:",
    POS_WEIGHT
)

Foreground fraction: 0.015173672542015501
Raw positive weight: 64.9036233470194
Used positive weight: 8.056278504807254


In [11]:
device = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


IMAGENET_MEAN = (
    classifier_metadata[
        "normalization"
    ]["mean"]
)

IMAGENET_STD = (
    classifier_metadata[
        "normalization"
    ]["std"]
)

MEAN_TENSOR = torch.tensor(
    IMAGENET_MEAN,
    dtype=torch.float32,
    device=device,
).view(
    1, 3, 1, 1
)

STD_TENSOR = torch.tensor(
    IMAGENET_STD,
    dtype=torch.float32,
    device=device,
).view(
    1, 3, 1, 1
)

Device: cuda:0
GPU: Tesla T4


In [12]:
def prepare_batch(
    images,
    masks,
):

    images = images.to(
        device,
        non_blocking=True,
    )

    masks = masks.to(
        device,
        non_blocking=True,
    )

    images = (
        images.float()
        / 65535.0
    )

    images = (
        images
        .unsqueeze(1)
        .repeat(
            1,
            3,
            1,
            1,
        )
    )

    images = (
        images
        - MEAN_TENSOR
    ) / STD_TENSOR

    masks = (
        masks.float()
        .unsqueeze(1)
    )

    return images, masks

In [13]:
classifier = resnet50(
    weights=None
)

classifier.fc = nn.Linear(
    classifier.fc.in_features,
    1,
)

classifier_checkpoint = torch.load(
    CLASSIFIER_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

classifier.load_state_dict(
    classifier_checkpoint[
        "model_state_dict"
    ]
)

print(
    "Classifier checkpoint epoch:",
    classifier_checkpoint["epoch"]
)

print(
    "Classifier validation ROC-AUC:",
    classifier_checkpoint[
        "val_metrics"
    ]["roc_auc"]
)

Classifier checkpoint epoch: 3
Classifier validation ROC-AUC: 0.8227387422360248


In [14]:
class DecoderBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
    ):

        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2,
        )

        self.conv = nn.Sequential(

            nn.Conv2d(
                out_channels
                + skip_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),

            nn.GroupNorm(
                8,
                out_channels,
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),

            nn.GroupNorm(
                8,
                out_channels,
            ),

            nn.ReLU(
                inplace=True
            ),
        )


    def forward(
        self,
        x,
        skip,
    ):

        x = self.up(x)

        x = torch.cat(
            [
                x,
                skip,
            ],
            dim=1,
        )

        return self.conv(x)

In [15]:
class ResNet50UNet(nn.Module):

    def __init__(
        self,
        backbone,
    ):

        super().__init__()


        # ----------------------------
        # Encoder
        # ----------------------------

        self.conv1 = backbone.conv1
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool

        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4


        # ----------------------------
        # Decoder
        # ----------------------------

        self.up4 = DecoderBlock(
            2048,
            1024,
            512,
        )

        self.up3 = DecoderBlock(
            512,
            512,
            256,
        )

        self.up2 = DecoderBlock(
            256,
            256,
            128,
        )

        self.up1 = DecoderBlock(
            128,
            64,
            64,
        )


        self.final_up = nn.Sequential(

            nn.ConvTranspose2d(
                64,
                32,
                kernel_size=2,
                stride=2,
            ),

            nn.GroupNorm(
                8,
                32,
            ),

            nn.ReLU(
                inplace=True
            ),
        )


        self.output = nn.Conv2d(
            32,
            1,
            kernel_size=1,
        )


        self.encoder_frozen = True


    def encode(
        self,
        x,
    ):

        e0 = self.relu(
            self.bn1(
                self.conv1(x)
            )
        )

        x = self.maxpool(e0)

        e1 = self.layer1(x)
        e2 = self.layer2(e1)
        e3 = self.layer3(e2)
        e4 = self.layer4(e3)

        return (
            e0,
            e1,
            e2,
            e3,
            e4,
        )


    def forward(
        self,
        x,
    ):

        if self.encoder_frozen:

            with torch.no_grad():

                (
                    e0,
                    e1,
                    e2,
                    e3,
                    e4,
                ) = self.encode(x)

        else:

            (
                e0,
                e1,
                e2,
                e3,
                e4,
            ) = self.encode(x)


        d4 = self.up4(
            e4,
            e3,
        )

        d3 = self.up3(
            d4,
            e2,
        )

        d2 = self.up2(
            d3,
            e1,
        )

        d1 = self.up1(
            d2,
            e0,
        )

        x = self.final_up(d1)

        return self.output(x)

In [16]:
model = ResNet50UNet(
    backbone=classifier
)

encoder_modules = [
    model.conv1,
    model.bn1,
    model.layer1,
    model.layer2,
    model.layer3,
    model.layer4,
]

for module in encoder_modules:

    for parameter in module.parameters():
        parameter.requires_grad = False


model = model.to(
    device
)


trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)


print(
    "Trainable parameters:",
    f"{trainable_parameters:,}"
)

print(
    "Total parameters:",
    f"{total_parameters:,}"
)

Trainable parameters: 17,392,449
Total parameters: 40,900,481


In [17]:
images, masks, _ = next(
    iter(train_loader)
)

images, masks = prepare_batch(
    images,
    masks,
)


with torch.inference_mode():

    logits = model(
        images
    )


print(
    "Input:",
    images.shape
)

print(
    "Output:",
    logits.shape
)

print(
    "Target:",
    masks.shape
)


assert logits.shape == masks.shape

print(
    "\nResNet-50 U-Net "
    "output shape verified."
)

Input: torch.Size([1, 3, 1024, 640])
Output: torch.Size([1, 1, 1024, 640])
Target: torch.Size([1, 1, 1024, 640])

ResNet-50 U-Net output shape verified.


In [18]:
def set_frozen_encoder_eval(
    model,
):

    model.conv1.eval()
    model.bn1.eval()

    model.layer1.eval()
    model.layer2.eval()
    model.layer3.eval()
    model.layer4.eval()

In [20]:
pos_weight_tensor = torch.tensor(
    [POS_WEIGHT],
    dtype=torch.float32,
    device=device,
)

bce_loss = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight_tensor
)


def soft_dice_per_image(
    logits,
    targets,
    smooth=1.0,
):

    probabilities = torch.sigmoid(
        logits
    )

    dimensions = (
        1,
        2,
        3,
    )

    intersection = (
        probabilities
        * targets
    ).sum(
        dim=dimensions
    )

    denominator = (
        probabilities.sum(
            dim=dimensions
        )
        +
        targets.sum(
            dim=dimensions
        )
    )

    return (
        (
            2.0 * intersection
            + smooth
        )
        /
        (
            denominator
            + smooth
        )
    )


def segmentation_loss(
    logits,
    targets,
):

    bce = bce_loss(
        logits,
        targets,
    )

    soft_dice = (
        soft_dice_per_image(
            logits,
            targets,
        ).mean()
    )

    dice_loss = (
        1.0
        - soft_dice
    )

    total = (
        BCE_WEIGHT * bce
        +
        DICE_WEIGHT * dice_loss
    )

    return {
        "total": total,
        "bce": bce,
        "dice_loss": dice_loss,
        "soft_dice": soft_dice,
    }

In [21]:
def hard_segmentation_metrics(
    logits,
    targets,
    threshold=0.5,
    smooth=1.0,
):

    probabilities = torch.sigmoid(
        logits
    )

    predictions = (
        probabilities
        >= threshold
    ).float()

    dimensions = (
        1,
        2,
        3,
    )

    intersection = (
        predictions
        * targets
    ).sum(
        dim=dimensions
    )

    predicted_pixels = (
        predictions.sum(
            dim=dimensions
        )
    )

    target_pixels = (
        targets.sum(
            dim=dimensions
        )
    )

    dice = (
        2.0 * intersection
        + smooth
    ) / (
        predicted_pixels
        + target_pixels
        + smooth
    )

    union = (
        predicted_pixels
        + target_pixels
        - intersection
    )

    iou = (
        intersection
        + smooth
    ) / (
        union
        + smooth
    )

    return {
        "dice": dice,
        "iou": iou,
    }

In [33]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    scaler,
):

    model.train()

    # Frozen ResNet encoder must stay in eval mode.
    keep_batchnorm_frozen(
        model
    )

    running = {
        "loss": 0.0,
        "soft_dice": 0.0,
        "hard_dice": 0.0,
        "iou": 0.0,
        "samples": 0,
    }

    for (
        images,
        masks,
        _,
    ) in tqdm(
        loader,
        desc="Training",
        leave=False,
    ):

        images, masks = prepare_batch(
            images,
            masks,
        )

        batch_size = (
            images.shape[0]
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(
                USE_AMP
                and device.type
                == "cuda"
            ),
        ):

            logits = model(
                images
            )

            losses = (
                segmentation_loss(
                    logits,
                    masks,
                )
            )

        scaler.scale(
            losses["total"]
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()

        with torch.no_grad():

            hard_metrics = (
                hard_segmentation_metrics(
                    logits,
                    masks,
                )
            )

        running["loss"] += (
            losses["total"].item()
            * batch_size
        )

        running["soft_dice"] += (
            losses["soft_dice"].item()
            * batch_size
        )

        running["hard_dice"] += (
            hard_metrics["dice"]
            .mean()
            .item()
            * batch_size
        )

        running["iou"] += (
            hard_metrics["iou"]
            .mean()
            .item()
            * batch_size
        )

        running["samples"] += (
            batch_size
        )

    samples = running[
        "samples"
    ]

    return {
        key: (
            running[key]
            / samples
        )
        for key in [
            "loss",
            "soft_dice",
            "hard_dice",
            "iou",
        ]
    }

In [24]:
def validate_one_epoch(
    model,
    loader,
):

    model.eval()

    running = {
        "loss": 0.0,
        "soft_dice": 0.0,
        "hard_dice": 0.0,
        "iou": 0.0,
        "samples": 0,
    }

    with torch.inference_mode():

        for (
            images,
            masks,
            _,
        ) in tqdm(
            loader,
            desc="Validation",
            leave=False,
        ):

            images, masks = (
                prepare_batch(
                    images,
                    masks,
                )
            )

            batch_size = (
                images.shape[0]
            )

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=(
                    USE_AMP
                    and device.type
                    == "cuda"
                ),
            ):

                logits = model(
                    images
                )

                losses = (
                    segmentation_loss(
                        logits,
                        masks,
                    )
                )

            hard_metrics = (
                hard_segmentation_metrics(
                    logits,
                    masks,
                )
            )

            running["loss"] += (
                losses["total"].item()
                * batch_size
            )

            running["soft_dice"] += (
                losses["soft_dice"].item()
                * batch_size
            )

            running["hard_dice"] += (
                hard_metrics["dice"]
                .mean()
                .item()
                * batch_size
            )

            running["iou"] += (
                hard_metrics["iou"]
                .mean()
                .item()
                * batch_size
            )

            running["samples"] += (
                batch_size
            )

    samples = running[
        "samples"
    ]

    return {
        key: (
            running[key]
            / samples
        )
        for key in [
            "loss",
            "soft_dice",
            "hard_dice",
            "iou",
        ]
    }

In [25]:
trainable_parameters = [
    parameter
    for parameter
    in model.parameters()
    if parameter.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=DECODER_LR,
    weight_decay=WEIGHT_DECAY,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(
        USE_AMP
        and device.type
        == "cuda"
    ),
)

print(
    "Decoder learning rate:",
    DECODER_LR
)

print(
    "AMP:",
    USE_AMP
)

Decoder learning rate: 0.0003
AMP: True


In [26]:
def save_checkpoint(
    path,
    epoch,
    model,
    optimizer,
    scaler,
    train_metrics,
    val_metrics,
):

    torch.save(
        {
            "stage": "decoder_only",

            "epoch": epoch,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "scaler_state_dict":
                scaler.state_dict(),

            "train_metrics":
                train_metrics,

            "val_metrics":
                val_metrics,

            "configuration": {
                "resolution": [
                    TARGET_HEIGHT,
                    TARGET_WIDTH,
                ],

                "batch_size":
                    BATCH_SIZE,

                "decoder_lr":
                    DECODER_LR,

                "weight_decay":
                    WEIGHT_DECAY,

                "bce_weight":
                    BCE_WEIGHT,

                "dice_weight":
                    DICE_WEIGHT,

                "positive_weight":
                    POS_WEIGHT,

                "encoder":
                    "classifier-trained ResNet-50",

                "encoder_frozen":
                    True,

                "seed":
                    SEED,
            },
        },
        path,
    )

In [27]:
history = []

best_val_soft_dice = (
    -float("inf")
)

epochs_without_improvement = 0


for epoch in range(
    1,
    MAX_EPOCHS + 1,
):

    print(
        f"\nEpoch "
        f"{epoch}/{MAX_EPOCHS}"
    )


    train_metrics = (
        train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            scaler=scaler,
        )
    )


    val_metrics = (
        validate_one_epoch(
            model=model,
            loader=val_loader,
        )
    )


    row = {
        "epoch": epoch,

        "train_loss":
            train_metrics[
                "loss"
            ],

        "train_soft_dice":
            train_metrics[
                "soft_dice"
            ],

        "train_hard_dice":
            train_metrics[
                "hard_dice"
            ],

        "train_iou":
            train_metrics[
                "iou"
            ],

        "val_loss":
            val_metrics[
                "loss"
            ],

        "val_soft_dice":
            val_metrics[
                "soft_dice"
            ],

        "val_hard_dice":
            val_metrics[
                "hard_dice"
            ],

        "val_iou":
            val_metrics[
                "iou"
            ],
    }


    history.append(
        row
    )


    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_PATH,
        index=False,
    )


    print(
        f"Train Loss: "
        f"{train_metrics['loss']:.4f} | "
        f"Soft Dice: "
        f"{train_metrics['soft_dice']:.4f} | "
        f"Hard Dice: "
        f"{train_metrics['hard_dice']:.4f}"
    )

    print(
        f"Val Loss: "
        f"{val_metrics['loss']:.4f} | "
        f"Soft Dice: "
        f"{val_metrics['soft_dice']:.4f} | "
        f"Hard Dice: "
        f"{val_metrics['hard_dice']:.4f} | "
        f"IoU: "
        f"{val_metrics['iou']:.4f}"
    )


    save_checkpoint(
        path=LATEST_CHECKPOINT_PATH,
        epoch=epoch,
        model=model,
        optimizer=optimizer,
        scaler=scaler,
        train_metrics=train_metrics,
        val_metrics=val_metrics,
    )


    improvement = (
        val_metrics[
            "soft_dice"
        ]
        >
        (
            best_val_soft_dice
            +
            EARLY_STOPPING_MIN_DELTA
        )
    )


    if improvement:

        best_val_soft_dice = (
            val_metrics[
                "soft_dice"
            ]
        )

        epochs_without_improvement = 0


        save_checkpoint(
            path=BEST_CHECKPOINT_PATH,
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            scaler=scaler,
            train_metrics=train_metrics,
            val_metrics=val_metrics,
        )


        print(
            "Saved new best decoder model "
            f"(val soft Dice = "
            f"{best_val_soft_dice:.4f})"
        )


    else:

        epochs_without_improvement += 1


        print(
            "No improvement for "
            f"{epochs_without_improvement} "
            "epoch(s)."
        )


    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):

        print(
            "\nEarly stopping triggered."
        )

        break


Epoch 1/8


Training:   0%|          | 0/1885 [00:00<?, ?it/s]

Validation:   0%|          | 0/408 [00:00<?, ?it/s]

Train Loss: 0.7176 | Soft Dice: 0.1044 | Hard Dice: 0.1561
Val Loss: 0.6604 | Soft Dice: 0.1705 | Hard Dice: 0.2377 | IoU: 0.1539
Saved new best decoder model (val soft Dice = 0.1705)

Epoch 2/8


Training:   0%|          | 0/1885 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/408 [00:00<?, ?it/s]

Train Loss: 0.5329 | Soft Dice: 0.3717 | Hard Dice: 0.4047
Val Loss: 0.5742 | Soft Dice: 0.3174 | Hard Dice: 0.3429 | IoU: 0.2418
No improvement for 1 epoch(s).

Epoch 8/8


Training:   0%|          | 0/1885 [00:00<?, ?it/s]

Validation:   0%|          | 0/408 [00:00<?, ?it/s]

Train Loss: 0.5014 | Soft Dice: 0.4120 | Hard Dice: 0.4412
Val Loss: 0.6488 | Soft Dice: 0.3089 | Hard Dice: 0.3150 | IoU: 0.2277
No improvement for 2 epoch(s).


In [28]:
# ============================================================
# STAGE B — LAYER4 + DECODER FINE-TUNING
# ============================================================

FINETUNE_MAX_EPOCHS = 6
FINETUNE_PATIENCE = 3
FINETUNE_MIN_DELTA = 1e-4

LAYER4_LR = 1e-5
FINETUNE_DECODER_LR = 1e-4

FINETUNE_BEST_PATH = (
    CHECKPOINT_DIR
    / "resnet50_unet_layer4_best.pth"
)

FINETUNE_LATEST_PATH = (
    CHECKPOINT_DIR
    / "resnet50_unet_layer4_latest.pth"
)

FINETUNE_HISTORY_PATH = (
    OUTPUT_DIR
    / "layer4_training_history.csv"
)

In [29]:
stage_a_checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=device,
    weights_only=False,
)

model.load_state_dict(
    stage_a_checkpoint[
        "model_state_dict"
    ]
)

print(
    "Loaded Stage-A epoch:",
    stage_a_checkpoint["epoch"]
)

print(
    "Stage-A best validation soft Dice:",
    stage_a_checkpoint[
        "val_metrics"
    ]["soft_dice"]
)

Loaded Stage-A epoch: 6
Stage-A best validation soft Dice: 0.3329135484901581


In [31]:
# Freeze everything first.
for parameter in model.parameters():
    parameter.requires_grad = False


# Unfreeze layer4.
for parameter in model.layer4.parameters():
    parameter.requires_grad = True


# Unfreeze decoder.
decoder_modules = [
    model.up4,
    model.up3,
    model.up2,
    model.up1,
    model.final_up,
    model.output,
]

for module in decoder_modules:
    for parameter in module.parameters():
        parameter.requires_grad = True


# Do not wrap the encoder in torch.no_grad anymore.
model.encoder_frozen = False


# Keep ALL BatchNorm parameters/statistics frozen.
# Batch size = 1 is too small to relearn reliable BN statistics.
for module in model.modules():

    if isinstance(
        module,
        nn.BatchNorm2d,
    ):

        module.eval()

        for parameter in module.parameters():
            parameter.requires_grad = False


layer4_trainable = sum(
    p.numel()
    for p in model.layer4.parameters()
    if p.requires_grad
)

decoder_trainable = sum(
    p.numel()
    for module in decoder_modules
    for p in module.parameters()
    if p.requires_grad
)

print(
    "Layer4 trainable parameters:",
    f"{layer4_trainable:,}"
)

print(
    "Decoder trainable parameters:",
    f"{decoder_trainable:,}"
)

print(
    "Total trainable:",
    f"{layer4_trainable + decoder_trainable:,}"
)

Layer4 trainable parameters: 14,942,208
Decoder trainable parameters: 17,392,449
Total trainable: 32,334,657


In [32]:
def keep_batchnorm_frozen(
    model,
):

    for module in model.modules():

        if isinstance(
            module,
            nn.BatchNorm2d,
        ):

            module.eval()

In [35]:
layer4_parameters = [
    parameter
    for parameter
    in model.layer4.parameters()
    if parameter.requires_grad
]


decoder_parameters = [
    parameter
    for module in decoder_modules
    for parameter in module.parameters()
    if parameter.requires_grad
]


optimizer = torch.optim.AdamW(
    [
        {
            "params": layer4_parameters,
            "lr": LAYER4_LR,
        },
        {
            "params": decoder_parameters,
            "lr": FINETUNE_DECODER_LR,
        },
    ],
    weight_decay=WEIGHT_DECAY,
)


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(
        USE_AMP
        and device.type == "cuda"
    ),
)


print(
    "Layer4 LR:",
    LAYER4_LR
)

print(
    "Decoder LR:",
    FINETUNE_DECODER_LR
)

Layer4 LR: 1e-05
Decoder LR: 0.0001


In [36]:
finetune_history = []

best_finetune_soft_dice = (
    stage_a_checkpoint[
        "val_metrics"
    ]["soft_dice"]
)

epochs_without_improvement = 0


for epoch in range(
    1,
    FINETUNE_MAX_EPOCHS + 1,
):

    print(
        f"\nFine-tune Epoch "
        f"{epoch}/{FINETUNE_MAX_EPOCHS}"
    )


    train_metrics = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        scaler=scaler,
    )


    val_metrics = validate_one_epoch(
        model=model,
        loader=val_loader,
    )


    row = {
        "epoch": epoch,

        "train_loss":
            train_metrics["loss"],

        "train_soft_dice":
            train_metrics["soft_dice"],

        "train_hard_dice":
            train_metrics["hard_dice"],

        "train_iou":
            train_metrics["iou"],

        "val_loss":
            val_metrics["loss"],

        "val_soft_dice":
            val_metrics["soft_dice"],

        "val_hard_dice":
            val_metrics["hard_dice"],

        "val_iou":
            val_metrics["iou"],
    }


    finetune_history.append(
        row
    )


    pd.DataFrame(
        finetune_history
    ).to_csv(
        FINETUNE_HISTORY_PATH,
        index=False,
    )


    print(
        f"Train Loss: "
        f"{train_metrics['loss']:.4f} | "
        f"Soft Dice: "
        f"{train_metrics['soft_dice']:.4f} | "
        f"Hard Dice: "
        f"{train_metrics['hard_dice']:.4f}"
    )

    print(
        f"Val Loss: "
        f"{val_metrics['loss']:.4f} | "
        f"Soft Dice: "
        f"{val_metrics['soft_dice']:.4f} | "
        f"Hard Dice: "
        f"{val_metrics['hard_dice']:.4f} | "
        f"IoU: "
        f"{val_metrics['iou']:.4f}"
    )


    # Latest checkpoint
    torch.save(
        {
            "stage": "layer4_finetune",
            "epoch": epoch,
            "model_state_dict":
                model.state_dict(),
            "optimizer_state_dict":
                optimizer.state_dict(),
            "scaler_state_dict":
                scaler.state_dict(),
            "train_metrics":
                train_metrics,
            "val_metrics":
                val_metrics,
        },
        FINETUNE_LATEST_PATH,
    )


    improvement = (
        val_metrics["soft_dice"]
        >
        best_finetune_soft_dice
        + FINETUNE_MIN_DELTA
    )


    if improvement:

        best_finetune_soft_dice = (
            val_metrics["soft_dice"]
        )

        epochs_without_improvement = 0


        torch.save(
            {
                "stage":
                    "layer4_finetune",

                "epoch":
                    epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "scaler_state_dict":
                    scaler.state_dict(),

                "train_metrics":
                    train_metrics,

                "val_metrics":
                    val_metrics,
            },
            FINETUNE_BEST_PATH,
        )


        print(
            "Saved new best layer4 model "
            f"(val soft Dice = "
            f"{best_finetune_soft_dice:.4f})"
        )


    else:

        epochs_without_improvement += 1

        print(
            "No improvement over "
            "Stage-A/best Stage-B model "
            f"for {epochs_without_improvement} "
            "epoch(s)."
        )


    if (
        epochs_without_improvement
        >= FINETUNE_PATIENCE
    ):

        print(
            "\nStage-B early stopping triggered."
        )

        break


Fine-tune Epoch 1/6


Training:   0%|          | 0/1885 [00:00<?, ?it/s]

Validation:   0%|          | 0/408 [00:00<?, ?it/s]

Train Loss: 0.5172 | Soft Dice: 0.3939 | Hard Dice: 0.4254
Val Loss: 0.5549 | Soft Dice: 0.3382 | Hard Dice: 0.3622 | IoU: 0.2571
Saved new best layer4 model (val soft Dice = 0.3382)

Fine-tune Epoch 2/6


Training:   0%|          | 0/1885 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/408 [00:00<?, ?it/s]

Train Loss: 0.3657 | Soft Dice: 0.5649 | Hard Dice: 0.5838
Val Loss: 0.6157 | Soft Dice: 0.3332 | Hard Dice: 0.3392 | IoU: 0.2468
No improvement over Stage-A/best Stage-B model for 2 epoch(s).

Fine-tune Epoch 6/6


Training:   0%|          | 0/1885 [00:00<?, ?it/s]

Validation:   0%|          | 0/408 [00:00<?, ?it/s]

Train Loss: 0.3450 | Soft Dice: 0.5890 | Hard Dice: 0.6066
Val Loss: 0.5931 | Soft Dice: 0.3263 | Hard Dice: 0.3344 | IoU: 0.2368
No improvement over Stage-A/best Stage-B model for 3 epoch(s).

Stage-B early stopping triggered.


In [37]:
from pathlib import Path
import json
import shutil
import pandas as pd


# ============================================================
# CREATE FINAL RESNET50-U-NET TRAINING ARTIFACT
# ============================================================

ARTIFACT_DIR = Path(
    "/kaggle/working/resnet50_unet_training_artifact"
)

if ARTIFACT_DIR.exists():
    shutil.rmtree(ARTIFACT_DIR)

ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# COPY BEST CHECKPOINTS
# ============================================================

assert BEST_CHECKPOINT_PATH.exists(), (
    "Stage-A best checkpoint not found."
)

assert FINETUNE_BEST_PATH.exists(), (
    "Stage-B best checkpoint not found."
)


shutil.copy2(
    BEST_CHECKPOINT_PATH,
    ARTIFACT_DIR
    / "resnet50_unet_decoder_best.pth",
)

shutil.copy2(
    FINETUNE_BEST_PATH,
    ARTIFACT_DIR
    / "resnet50_unet_layer4_best.pth",
)


# ============================================================
# COPY TRAINING HISTORIES
# ============================================================

assert HISTORY_PATH.exists()

assert FINETUNE_HISTORY_PATH.exists()


shutil.copy2(
    HISTORY_PATH,
    ARTIFACT_DIR
    / "decoder_training_history.csv",
)

shutil.copy2(
    FINETUNE_HISTORY_PATH,
    ARTIFACT_DIR
    / "layer4_training_history.csv",
)


# ============================================================
# LOAD HISTORIES
# ============================================================

stage_a_history = pd.read_csv(
    HISTORY_PATH
)

stage_b_history = pd.read_csv(
    FINETUNE_HISTORY_PATH
)


stage_a_best_row = (
    stage_a_history.loc[
        stage_a_history[
            "val_soft_dice"
        ].idxmax()
    ]
)

stage_b_best_row = (
    stage_b_history.loc[
        stage_b_history[
            "val_soft_dice"
        ].idxmax()
    ]
)


# ============================================================
# SAVE TRAINING SUMMARY
# ============================================================

training_summary = {

    "architecture":
        "ResNet50UNet",

    "encoder":
        "ResNet-50",

    "encoder_initialization":
        (
            "ImageNet pretrained ResNet-50 "
            "fine-tuned on CBIS-DDSM "
            "malignancy classification"
        ),

    "input_resolution": [
        TARGET_HEIGHT,
        TARGET_WIDTH,
    ],

    "input_channels": 3,

    "segmentation_output_channels": 1,

    "batch_size":
        BATCH_SIZE,

    "loss": {
        "bce_weight":
            BCE_WEIGHT,

        "dice_weight":
            DICE_WEIGHT,

        "positive_weight":
            POS_WEIGHT,
    },

    "seed":
        SEED,


    # --------------------------------------------------------
    # Stage A
    # --------------------------------------------------------

    "stage_a": {

        "name":
            "decoder_only",

        "encoder_frozen":
            True,

        "decoder_lr":
            DECODER_LR,

        "weight_decay":
            WEIGHT_DECAY,

        "best_epoch":
            int(
                stage_a_best_row[
                    "epoch"
                ]
            ),

        "best_validation_soft_dice":
            float(
                stage_a_best_row[
                    "val_soft_dice"
                ]
            ),

        "validation_hard_dice_at_0_5":
            float(
                stage_a_best_row[
                    "val_hard_dice"
                ]
            ),

        "validation_iou_at_0_5":
            float(
                stage_a_best_row[
                    "val_iou"
                ]
            ),

        "checkpoint":
            (
                "resnet50_unet_"
                "decoder_best.pth"
            ),
    },


    # --------------------------------------------------------
    # Stage B
    # --------------------------------------------------------

    "stage_b": {

        "name":
            "layer4_finetune",

        "layer4_unfrozen":
            True,

        "batchnorm_frozen":
            True,

        "layer4_lr":
            LAYER4_LR,

        "decoder_lr":
            FINETUNE_DECODER_LR,

        "weight_decay":
            WEIGHT_DECAY,

        "best_epoch":
            int(
                stage_b_best_row[
                    "epoch"
                ]
            ),

        "best_validation_soft_dice":
            float(
                stage_b_best_row[
                    "val_soft_dice"
                ]
            ),

        "validation_hard_dice_at_0_5":
            float(
                stage_b_best_row[
                    "val_hard_dice"
                ]
            ),

        "validation_iou_at_0_5":
            float(
                stage_b_best_row[
                    "val_iou"
                ]
            ),

        "checkpoint":
            (
                "resnet50_unet_"
                "layer4_best.pth"
            ),
    },


    # --------------------------------------------------------
    # Final model selection
    # --------------------------------------------------------

    "selected_model":
        "stage_b",

    "selected_checkpoint":
        (
            "resnet50_unet_"
            "layer4_best.pth"
        ),

    "selected_validation_soft_dice":
        float(
            stage_b_best_row[
                "val_soft_dice"
            ]
        ),

    "selected_validation_hard_dice_at_0_5":
        float(
            stage_b_best_row[
                "val_hard_dice"
            ]
        ),

    "selected_validation_iou_at_0_5":
        float(
            stage_b_best_row[
                "val_iou"
            ]
        ),

    "segmentation_threshold_selected":
        False,

    "test_set_used":
        False,

    "notes":
        (
            "Stage A trained only the decoder "
            "with the classifier-trained "
            "ResNet-50 encoder frozen. "
            "Stage B unfroze layer4 using a "
            "smaller learning rate while "
            "keeping BatchNorm frozen. "
            "Final checkpoint selected using "
            "validation soft Dice. "
            "Threshold selection and final "
            "test evaluation are deferred "
            "to the evaluation notebook."
        ),
}


SUMMARY_PATH = (
    ARTIFACT_DIR
    / "training_summary.json"
)


with open(
    SUMMARY_PATH,
    "w",
) as f:

    json.dump(
        training_summary,
        f,
        indent=2,
    )


# ============================================================
# VERIFY ARTIFACT CONTENTS
# ============================================================

print(
    "ResNet50-U-Net training artifact:"
)

print("-" * 60)


for path in sorted(
    ARTIFACT_DIR.rglob("*")
):

    if path.is_file():

        size_mb = (
            path.stat().st_size
            / (1024 ** 2)
        )

        print(
            f"{path.relative_to(ARTIFACT_DIR)}"
            f"  ({size_mb:.2f} MB)"
        )


# ============================================================
# CREATE ZIP
# ============================================================

ZIP_BASE = (
    "/kaggle/working/"
    "resnet50_unet_training_artifact"
)

zip_path = shutil.make_archive(
    ZIP_BASE,
    "zip",
    ARTIFACT_DIR,
)


zip_size_mb = (
    Path(zip_path).stat().st_size
    / (1024 ** 2)
)


print("\n" + "=" * 60)

print(
    "ZIP created:"
)

print(
    zip_path
)

print(
    f"ZIP size: "
    f"{zip_size_mb:.2f} MB"
)

print("=" * 60)

ResNet50-U-Net training artifact:
------------------------------------------------------------
decoder_training_history.csv  (0.00 MB)
layer4_training_history.csv  (0.00 MB)
resnet50_unet_decoder_best.pth  (289.07 MB)
resnet50_unet_layer4_best.pth  (403.08 MB)
training_summary.json  (0.00 MB)

ZIP created:
/kaggle/working/resnet50_unet_training_artifact.zip
ZIP size: 643.50 MB
